# 5- Model Deployment and Real-Time Prediction

**NB: For the deployment phase, we will proceed with the first approach. Based on the trained models, **`Logistic Regression`** achieved the highest accuracy; therefore, it has been selected for production deployment.**

In [73]:
log_reg.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [75]:
import joblib

# Save model artifacts
joblib.dump(log_reg, 'logistic_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [74]:
import os
print(os.getcwd())

/home/ec2-user/SageMaker


In [76]:
# Ensure inference.py file is in the same directory as this script.
# Then, compress inference.py along with the model files.
!tar -czvf model_artifacts.tar.gz inference.py logistic_model.pkl tfidf_vectorizer.pkl

inference.py
logistic_model.pkl
tfidf_vectorizer.pkl


In [77]:
# Upload the tar.gz file to S3
import boto3
s3 = boto3.client('s3')
s3.upload_file('model_artifacts.tar.gz', 'shenawy-ml-text', 'models/model_artifacts.tar.gz')
print("Artifacts uploaded to S3 successfully!")

Artifacts uploaded to S3 successfully!


In [ ]:
import boto3

sm = boto3.client('sagemaker')

endpoint_name = 'text-classifier-endpoint'

# Delete the endpoint and its config
sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_name)

print("Deleted old endpoint and config.")

In [80]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import get_execution_role, Session

sagemaker_session = Session()
role = get_execution_role()

model = SKLearnModel(
    model_data='s3://shenawy-ml-text/models/model_artifacts.tar.gz',
    role=role,
    entry_point='inference.py',
    framework_version='0.23-1',
    sagemaker_session=sagemaker_session
)

predictor = model.deploy(
    instance_type='ml.t2.medium',
    initial_instance_count=1,
    endpoint_name='text-classifier-endpoint-v3'
)

[04/13/25 00:55:09] INFO     Creating model with name:                                              ]8;id=879473;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=202014;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#4094\4094]8;;\
                             sagemaker-scikit-learn-2025-04-13-00-55-09-680                                        

[04/13/25 00:55:10] INFO     Creating endpoint-config with name text-classifier-endpoint-v3         ]8;id=2745;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=489205;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#6019\6019]8;;\

                    INFO     Creating endpoint with name text-classifier-endpoint-v3                ]8;id=563405;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=317359;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#4841\4841]8;;\

-----------!

In [93]:
import requests

# Define the API URL (replace with your correct invoke URL)
api_url = "https://a6p7fwx9xe.execute-api.us-east-1.amazonaws.com/prod/predict"

# Sample input text you want to classify
input_text = {"text": "This is the sentence I want to classify"}

# Make the POST request to the API
response = requests.post(api_url, json=input_text)

# Print the response from the API (prediction result)
print(response.json())

{'statusCode': 400, 'body': '{"error": "No body found in the request"}'}
